# MCP (Model Context Protocol) for Computational Toxicology
### A Simple Step-by-Step Guide to Connecting AI Agents with Toxicology Tools

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

## What is MCP?

**MCP (Model Context Protocol)** is an open standard (Anthropic, 2024) that lets AI models
like Claude connect to external tools, databases, and services in a standardised way.

Think of it as a **universal plug** between AI and scientific software:

```
Without MCP                          With MCP
------------------                   ------------------
AI model                             AI model
  |                                    |
  |-- custom code for RDKit            |-- MCP protocol
  |-- custom code for ChEMBL           |-- MCP protocol
  |-- custom code for PubChem          |-- MCP protocol
  |-- custom code for ADMET DB         |-- MCP protocol
  (each needs unique integration)      (one standard for all)
```

## Why MCP matters for toxicology

| Problem | MCP Solution |
|---------|-------------|
| AI cannot run RDKit to compute descriptors | MCP exposes RDKit as a tool |
| AI cannot query ChEMBL database | MCP exposes ChEMBL API as a resource |
| AI cannot read SDF files | MCP provides file access tools |
| AI cannot call ADMET models | MCP wraps models as callable tools |
| Results not reproducible | MCP logs every tool call with inputs/outputs |

## What you will learn

| Step | What | Complexity |
|------|------|----------|
| 1 | What MCP is and how it works | Concept |
| 2 | Install and run your first MCP server | Beginner |
| 3 | Build a simple toxicology MCP server | Beginner |
| 4 | Expose RDKit as MCP tools | Intermediate |
| 5 | Connect to ChEMBL and PubChem via MCP | Intermediate |
| 6 | ADMET prediction tools via MCP | Intermediate |
| 7 | Multi-tool toxicology agent | Advanced |
| 8 | Connect Claude to your MCP server | Advanced |
| 9 | Production patterns and best practices | Expert |
| 10 | Complete toxicology MCP toolkit | Reference |

---
## Section 1 -- How MCP Works: Core Concepts

MCP has three building blocks: **Servers**, **Clients**, and **Protocol Messages**.

```
MCP CLIENT (Claude / AI agent)          MCP SERVER (your Python code)
--------------------------------        --------------------------------
1. Discover: 'what tools exist?'  --->  return list of tools
2. Call: tool_name + arguments    --->  run tool, return result
3. Read: 'give me resource X'     --->  return file/database content
4. Subscribe: stream updates      --->  push real-time data
```

### The three things an MCP server exposes

| Primitive | What it is | Toxicology example |
|-----------|-----------|-------------------|
| **Tools** | Functions the AI can call | compute_descriptors(smiles) |
| **Resources** | Data the AI can read | toxicology_database.sqlite |
| **Prompts** | Template prompts | iata_assessment_template |

### Transport methods (how client talks to server)

```
STDIO transport (simplest -- default for local tools)
  Claude --> stdin --> your_server.py --> stdout --> Claude

HTTP/SSE transport (for web services, remote databases)
  Claude --> HTTP POST --> http://localhost:8000/mcp --> Claude
```

### The MCP message format (JSON-RPC 2.0)

```json
// Client sends:
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "compute_admet",
    "arguments": {"smiles": "CC(=O)Oc1ccccc1C(=O)O"}
  }
}

// Server responds:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [{"type": "text", "text": "{MW: 180.16, LogP: 1.19, ...}"}]
  }
}
```

In [ ]:
# Section 1: No code to run -- this is a concepts section
# Let's verify we have the tools we need

import subprocess, sys

def check_package(name, import_name=None):
    import_name = import_name or name
    try:
        __import__(import_name)
        return True
    except ImportError:
        return False

packages = [
    ('rdkit',          'rdkit'),
    ('mcp',            'mcp'),
    ('fastapi',        'fastapi'),
    ('httpx',          'httpx'),
    ('pydantic',       'pydantic'),
    ('numpy',          'numpy'),
    ('pandas',         'pandas'),
]

print('Package availability check:')
for pkg_name, import_name in packages:
    available = check_package(pkg_name, import_name)
    status = 'OK' if available else 'NOT INSTALLED'
    print(f'  {pkg_name:15s}: {status}')

print()
print('To install MCP:')
print('  pip install mcp')
print()
print('To install all dependencies:')
print('  pip install mcp rdkit fastapi httpx pydantic numpy pandas')

---
## Section 2 -- Your First MCP Server in 20 Lines

The fastest way to understand MCP is to build the simplest possible server.
This one has a single tool: `hello_toxicology`.

**Step 1:** Install the MCP SDK
```bash
pip install mcp
```

**Step 2:** Write the server (shown below)

**Step 3:** Run it
```bash
python my_first_server.py
```

**Step 4:** Connect Claude Desktop (or any MCP client) to it

In [ ]:
# ── Write the simplest possible MCP server to disk ─────────────────────────
import os

server_code = '''
#!/usr/bin/env python3
# my_first_server.py -- simplest possible MCP server
# Run: python my_first_server.py

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types

# 1. Create the server
server = Server("hello-toxicology")

# 2. Define a tool
@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name="hello_toxicology",
            description="Says hello and returns a welcome message about toxicology",
            inputSchema={
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Your name"}
                },
                "required": ["name"]
            }
        )
    ]

# 3. Handle tool calls
@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "hello_toxicology":
        user_name = arguments.get("name", "scientist")
        message = (
            f"Hello {user_name}! Welcome to computational toxicology via MCP.\n"
            f"This server can run toxicology analyses for you.\n"
            f"Available soon: ADMET prediction, hERG screening, Ames QSAR."
        )
        return [types.TextContent(type="text", text=message)]
    raise ValueError(f"Unknown tool: {name}")

# 4. Run (STDIO transport -- simplest, works locally)
async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            server.create_initialization_options()
        )

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
'''

# Save to file
with open('my_first_server.py', 'w') as f:
    f.write(server_code)
print('Saved: my_first_server.py')
print()
print('Key parts explained:')
print('  Server(name)        -- creates a named MCP server')
print('  @server.list_tools  -- decorator: tells client what tools exist')
print('  @server.call_tool   -- decorator: handles when client calls a tool')
print('  stdio_server()      -- STDIO transport (local use)')
print('  server.run()        -- starts the server loop')
print()
print('To connect to Claude Desktop, add to ~/Library/Application Support/Claude/claude_desktop_config.json:')
print('{')
print('  "mcpServers": {')
print('    "my-tox-server": {')
print('      "command": "python",')
print('      "args": ["/path/to/my_first_server.py"]')
print('    }')
print('  }')
print('}')

---
## Section 3 -- RDKit Cheminformatics as MCP Tools

Now we expose real toxicology functions as MCP tools. This server gives Claude
the ability to compute molecular descriptors from SMILES strings.

### Pattern: one Python function = one MCP tool
```
Python function               MCP tool
------------------            ------------------
def compute_admet(smiles):    name: compute_admet
    ...                       description: Compute ADMET properties
    return dict               inputSchema: {smiles: string}
```

In [ ]:
# ── The RDKit toxicology functions (the science) ───────────────────────────
# These are the actual functions that do the work.
# In MCP, we wrap these functions as tools.

import warnings; warnings.filterwarnings('ignore')
import json

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
    from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False
    print('RDKit not available -- install: pip install rdkit')

def compute_admet_properties(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None:
        return {'error': f'Invalid SMILES: {smiles}'}
    return {
        'MW':       round(Descriptors.MolWt(mol), 2),
        'LogP':     round(Descriptors.MolLogP(mol), 3),
        'TPSA':     round(Descriptors.TPSA(mol), 1),
        'HBD':      rdMolDescriptors.CalcNumHBD(mol),
        'HBA':      rdMolDescriptors.CalcNumHBA(mol),
        'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'QED':      round(QED.qed(mol), 3),
        'Ro5_violations': sum([
            Descriptors.MolWt(mol) > 500,
            Descriptors.MolLogP(mol) > 5,
            rdMolDescriptors.CalcNumHBD(mol) > 5,
            rdMolDescriptors.CalcNumHBA(mol) > 10,
        ]),
        'oral_bioavailability': 'Likely' if sum([
            Descriptors.MolWt(mol) > 500,
            Descriptors.MolLogP(mol) > 5,
            rdMolDescriptors.CalcNumHBD(mol) > 5,
            rdMolDescriptors.CalcNumHBA(mol) > 10,
        ]) <= 1 else 'Poor',
    }

def screen_structural_alerts(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None:
        return {'error': f'Invalid SMILES: {smiles}'}
    alerts = {
        'Nitrosamine':   '[N;!$(N=O)]-N=O',
        'Aromatic_nitro':'c[N+](=O)[O-]',
        'Ar_amine':      '[NH2]c',
        'Michael_acc':   '[$(C=CC=O),$(C=CS)]',
        'Epoxide':       '[C;R0]1OC1',
        'Diazonium':     '[#6][N+]#N',
        'Furan':         'c1ccoc1',
        'Hydrazine':     '[NH2]N',
    }
    hits = []
    for name, smarts in alerts.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt and mol.HasSubstructMatch(patt):
            hits.append(name)
    class1 = [h for h in hits if h in ['Nitrosamine']]
    class2 = [h for h in hits if h not in class1]
    return {
        'alerts_found': hits,
        'n_alerts': len(hits),
        'ich_m7_class': (
            'Class 1 (known human mutagen)' if class1
            else 'Class 2 (animal mutagen)' if class2
            else 'Class 5 (no alert)'
        ),
        'genotox_concern': len(hits) > 0,
    }

def screen_pains(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None:
        return {'error': f'Invalid SMILES: {smiles}'}
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog(params)
    entry = catalog.GetFirstMatch(mol)
    if entry:
        return {'pains_alert': True, 'pattern': entry.GetDescription(),
                'recommendation': 'Likely false positive in assays -- flag for review'}
    return {'pains_alert': False, 'status': 'Clean -- no PAINS detected'}

def compute_similarity(smiles1: str, smiles2: str) -> dict:
    mol1 = Chem.MolFromSmiles(smiles1) if RDKIT_OK else None
    mol2 = Chem.MolFromSmiles(smiles2) if RDKIT_OK else None
    if mol1 is None or mol2 is None:
        return {'error': 'One or both SMILES invalid'}
    from rdkit import DataStructs
    fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, 2, 2048)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, 2, 2048)
    tc = DataStructs.TanimotoSimilarity(fp1, fp2)
    return {
        'tanimoto_similarity': round(tc, 4),
        'interpretation': (
            'Nearly identical' if tc > 0.99
            else 'Very similar' if tc > 0.85
            else 'Similar' if tc > 0.60
            else 'Related' if tc > 0.40
            else 'Dissimilar'
        ),
        'read_across_candidate': tc >= 0.40,
    }

# Test the functions directly first
test_smiles = 'CC(=O)Oc1ccccc1C(=O)O'  # Aspirin
print('Testing functions on Aspirin:')
print()
print('ADMET:', json.dumps(compute_admet_properties(test_smiles), indent=2))
print()
print('Alerts:', json.dumps(screen_structural_alerts(test_smiles), indent=2))
print()
print('PAINS:', json.dumps(screen_pains(test_smiles), indent=2))

In [ ]:
# ── Now wrap these functions as an MCP server ────────────────────────────────
# Each function becomes a tool the AI can call

rdkit_server_code = '''
#!/usr/bin/env python3
# rdkit_mcp_server.py -- RDKit cheminformatics as MCP tools
# Run: python rdkit_mcp_server.py

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import json
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog

server = Server("rdkit-toxicology")

# ── Step 1: Declare all tools ─────────────────────────────────────────────
@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name="compute_admet",
            description="Compute ADMET physicochemical properties for a molecule: MW, LogP, TPSA, HBD, HBA, QED, Ro5",
            inputSchema={
                "type": "object",
                "properties": {
                    "smiles": {"type": "string", "description": "SMILES string of the molecule"}
                },
                "required": ["smiles"]
            }
        ),
        types.Tool(
            name="screen_alerts",
            description="Screen a molecule for ICH M7 structural alerts (genotoxicity flags for regulatory assessment)",
            inputSchema={
                "type": "object",
                "properties": {
                    "smiles": {"type": "string", "description": "SMILES string"}
                },
                "required": ["smiles"]
            }
        ),
        types.Tool(
            name="screen_pains",
            description="Screen for PAINS (pan-assay interference compounds) structural alerts",
            inputSchema={
                "type": "object",
                "properties": {
                    "smiles": {"type": "string", "description": "SMILES string"}
                },
                "required": ["smiles"]
            }
        ),
        types.Tool(
            name="compute_similarity",
            description="Compute Tanimoto similarity (ECFP4) between two molecules for read-across assessment",
            inputSchema={
                "type": "object",
                "properties": {
                    "smiles1": {"type": "string", "description": "First molecule SMILES"},
                    "smiles2": {"type": "string", "description": "Second molecule SMILES"}
                },
                "required": ["smiles1", "smiles2"]
            }
        ),
    ]

# ── Step 2: Handle tool calls ─────────────────────────────────────────────
@server.call_tool()
async def call_tool(name: str, arguments: dict):
    try:
        if name == "compute_admet":
            result = compute_admet_properties(arguments["smiles"])
        elif name == "screen_alerts":
            result = screen_structural_alerts(arguments["smiles"])
        elif name == "screen_pains":
            result = screen_pains(arguments["smiles"])
        elif name == "compute_similarity":
            result = compute_similarity(arguments["smiles1"], arguments["smiles2"])
        else:
            result = {"error": f"Unknown tool: {name}"}
        return [types.TextContent(type="text", text=json.dumps(result, indent=2))]
    except Exception as e:
        return [types.TextContent(type="text", text=json.dumps({"error": str(e)}))]


# --- Helper functions (same as notebook) ---
def compute_admet_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"error": f"Invalid SMILES: {smiles}"}
    return {
        "MW": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 3),
        "TPSA": round(Descriptors.TPSA(mol), 1),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "QED": round(QED.qed(mol), 3),
        "Ro5_violations": sum([Descriptors.MolWt(mol)>500, Descriptors.MolLogP(mol)>5,
                                rdMolDescriptors.CalcNumHBD(mol)>5, rdMolDescriptors.CalcNumHBA(mol)>10]),
    }

def screen_structural_alerts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"error": f"Invalid SMILES"}
    alerts = {"Nitrosamine": "[N;!$(N=O)]-N=O", "Aromatic_nitro": "c[N+](=O)[O-]",
              "Ar_amine": "[NH2]c", "Michael_acc": "[$(C=CC=O)]", "Epoxide": "[C;R0]1OC1"}
    hits = [k for k, v in alerts.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
    return {"alerts_found": hits, "n_alerts": len(hits),
            "ich_m7_class": "Class 1" if "Nitrosamine" in hits else ("Class 2" if hits else "Class 5"),
            "genotox_concern": len(hits) > 0}

def screen_pains(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"error": "Invalid SMILES"}
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog(params)
    entry = catalog.GetFirstMatch(mol)
    if entry: return {"pains_alert": True, "pattern": entry.GetDescription()}
    return {"pains_alert": False, "status": "Clean"}

def compute_similarity(smiles1, smiles2):
    m1, m2 = Chem.MolFromSmiles(smiles1), Chem.MolFromSmiles(smiles2)
    if not m1 or not m2: return {"error": "Invalid SMILES"}
    fp1 = AllChem.GetMorganFingerprintAsBitVect(m1, 2, 2048)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(m2, 2, 2048)
    tc = DataStructs.TanimotoSimilarity(fp1, fp2)
    return {"tanimoto": round(tc, 4), "read_across_candidate": tc >= 0.40}

# ── Step 3: Run server ────────────────────────────────────────────────────
async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(read_stream, write_stream,
                        server.create_initialization_options())

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
'''

with open('rdkit_mcp_server.py', 'w') as f:
    f.write(rdkit_server_code)
print('Saved: rdkit_mcp_server.py')
print()
print('This server exposes 4 RDKit tools to Claude:')
print('  1. compute_admet  -- MW, LogP, TPSA, HBD, HBA, QED, Ro5')
print('  2. screen_alerts  -- ICH M7 structural alert screening')
print('  3. screen_pains   -- PAINS interference compound detection')
print('  4. compute_similarity -- Tanimoto for read-across')

---
## Section 4 -- Exposing Databases as MCP Resources

MCP **Resources** are read-only data the AI can access -- files, databases, URLs.
This section shows how to expose a toxicology database as an MCP resource.

### Resources vs Tools

```
TOOLS                            RESOURCES
-----------                      -------------
- AI calls them (active)         - AI reads them (passive)
- Take arguments                 - Have a URI address
- Run computations               - Return data
- Example: compute_admet(smi)    - Example: tox://compounds/aspirin
```

### URI scheme for toxicology
```
tox://compound/{smiles}          ADMET data for a compound
tox://database/dilirank          FDA DILIrank database
tox://reference/ich_m7           ICH M7 guidelines text
tox://assay/{assay_id}           ToxCast assay result
```

In [ ]:
# ── Build a small in-memory toxicology database ──────────────────────────────
import json

# Simulated reference database (in production: SQLite or PostgreSQL)
TOX_DATABASE = {
    'aspirin': {
        'name': 'Aspirin', 'cas': '50-78-2',
        'smiles': 'CC(=O)Oc1ccccc1C(=O)O',
        'mw': 180.16, 'logp': 1.19, 'solubility': 'Moderate',
        'ames': 'Negative', 'herg_risk': 'Low', 'dili': 'Less-concern',
        'ld50_rat_oral_mg_kg': 200, 'ghs_class': 'Category 4',
        'therapeutic_class': 'NSAID analgesic',
    },
    'diclofenac': {
        'name': 'Diclofenac', 'cas': '15307-86-5',
        'smiles': 'O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl',
        'mw': 296.15, 'logp': 3.96, 'solubility': 'Low',
        'ames': 'Negative', 'herg_risk': 'Medium', 'dili': 'Most-concern',
        'ld50_rat_oral_mg_kg': 150, 'ghs_class': 'Category 4',
        'therapeutic_class': 'NSAID analgesic',
    },
    'caffeine': {
        'name': 'Caffeine', 'cas': '58-08-2',
        'smiles': 'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
        'mw': 194.19, 'logp': -0.07, 'solubility': 'High',
        'ames': 'Negative', 'herg_risk': 'Low', 'dili': 'No-concern',
        'ld50_rat_oral_mg_kg': 367, 'ghs_class': 'Category 5',
        'therapeutic_class': 'Stimulant / Xanthine',
    },
    'cisapride': {
        'name': 'Cisapride', 'cas': '81098-60-4',
        'smiles': 'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1',
        'mw': 465.95, 'logp': 2.88, 'solubility': 'Low',
        'ames': 'Negative', 'herg_risk': 'HIGH', 'dili': 'Most-concern',
        'ld50_rat_oral_mg_kg': 200, 'ghs_class': 'Category 4',
        'therapeutic_class': 'Prokinetic (withdrawn -- cardiac risk)',
    },
    'ndma': {
        'name': 'NDMA (N-Nitrosodimethylamine)', 'cas': '62-75-9',
        'smiles': 'CN(C)N=O',
        'mw': 74.08, 'logp': -0.57, 'solubility': 'High',
        'ames': 'Positive', 'herg_risk': 'Low', 'dili': 'N/A',
        'ld50_rat_oral_mg_kg': 27, 'ghs_class': 'Category 2',
        'iarc_class': 'Group 2A (probable carcinogen)',
        'ich_m7_class': 'Class 1 (known human mutagen)',
        'acceptable_intake_ng_day': 0.096,
    },
}

print('Toxicology database loaded:')
for key, info in TOX_DATABASE.items():
    print(f'  {key:12s}: {info["name"]:35s} DILI={info["dili"]}')

In [ ]:
# ── MCP server with Resources + Tools ───────────────────────────────────────

resource_server_code = '''
#!/usr/bin/env python3
# tox_database_server.py -- database as MCP resources + search tools

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import json

server = Server("tox-database")
TOX_DB = {}  # populated at startup -- see full version in notebook

# ── Resources: read-only data the AI can browse ────────────────────────────
@server.list_resources()
async def list_resources():
    # Tell the AI what resources exist
    resources = [
        types.Resource(
            uri="tox://database/summary",
            name="Toxicology Database Summary",
            description="Overview of all compounds in the database",
            mimeType="application/json"
        )
    ]
    # Add one resource per compound
    for compound_id, info in TOX_DB.items():
        resources.append(types.Resource(
            uri=f"tox://compound/{compound_id}",
            name=f"{info[name]} toxicology profile",
            description=f"Full toxicology data for {info[name]} (CAS {info.get(cas, N/A)})",
            mimeType="application/json"
        ))
    return resources

@server.read_resource()
async def read_resource(uri: str):
    # Return data for a requested resource URI
    if uri == "tox://database/summary":
        summary = {
            "total_compounds": len(TOX_DB),
            "compounds": [
                {"id": k, "name": v["name"], "dili": v.get("dili", "N/A")}
                for k, v in TOX_DB.items()
            ]
        }
        return types.ReadResourceResult(
            contents=[types.TextResourceContents(
                uri=uri,
                mimeType="application/json",
                text=json.dumps(summary, indent=2)
            )]
        )
    if uri.startswith("tox://compound/"):
        compound_id = uri.split("/")[-1]
        if compound_id in TOX_DB:
            return types.ReadResourceResult(
                contents=[types.TextResourceContents(
                    uri=uri,
                    mimeType="application/json",
                    text=json.dumps(TOX_DB[compound_id], indent=2)
                )]
            )
    raise ValueError(f"Unknown resource: {uri}")

# ── Tools: active search functions ────────────────────────────────────────
@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name="search_compound",
            description="Search for a compound in the toxicology database by name or CAS number",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Compound name or CAS number"}
                },
                "required": ["query"]
            }
        ),
        types.Tool(
            name="get_dili_compounds",
            description="Get all compounds with a specific DILI (Drug-Induced Liver Injury) classification",
            inputSchema={
                "type": "object",
                "properties": {
                    "dili_class": {
                        "type": "string",
                        "enum": ["Most-concern", "Less-concern", "No-concern"],
                        "description": "DILIrank classification"
                    }
                },
                "required": ["dili_class"]
            }
        )
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "search_compound":
        query = arguments["query"].lower()
        matches = [
            {"id": k, **v}
            for k, v in TOX_DB.items()
            if query in k.lower() or query in v.get("name", "").lower()
            or query in v.get("cas", "").lower()
        ]
        result = {"n_matches": len(matches), "compounds": matches}
    elif name == "get_dili_compounds":
        dili_class = arguments["dili_class"]
        matches = [{"id": k, "name": v["name"], "dili": v["dili"]}
                   for k, v in TOX_DB.items() if v.get("dili") == dili_class]
        result = {"dili_class": dili_class, "n_compounds": len(matches), "compounds": matches}
    else:
        result = {"error": f"Unknown tool: {name}"}
    return [types.TextContent(type="text", text=json.dumps(result, indent=2))]

async def main():
    async with stdio_server() as (r, w):
        await server.run(r, w, server.create_initialization_options())

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
'''

with open('tox_database_server.py', 'w') as f:
    f.write(resource_server_code)
print('Saved: tox_database_server.py')
print()
print('This server exposes:')
print('  Resources (read):  tox://database/summary, tox://compound/{id}')
print('  Tools (call):      search_compound, get_dili_compounds')

# Demonstrate the search logic
print()
print('Search demo:')
query = 'aspirin'
matches = [{k: v} for k, v in TOX_DATABASE.items() if query in k.lower()]
print(f'  search("{query}"): found {len(matches)} match(es)')
dili_high = [k for k, v in TOX_DATABASE.items() if v.get('dili') == 'Most-concern']
print(f'  DILI Most-concern: {dili_high}')

---
## Section 5 -- HTTP/SSE Transport: Connecting Remote Services

STDIO transport is great for local tools. For **remote databases, web APIs, or
shared team servers**, use HTTP with SSE (Server-Sent Events) transport.

```
STDIO transport              HTTP/SSE transport
------------------           ------------------
Local only                   Network accessible
Single client                Multiple clients
Subprocess                   Web server (FastAPI)
No authentication            Can add API keys
Simplest                     Production standard
```

**When to use HTTP transport:**
- Team shares one server running toxicology models
- Server has GPU for ML predictions
- Connecting to real PubChem / ChEMBL APIs
- Deploying to cloud

In [ ]:
# ── HTTP/SSE MCP server using FastAPI ────────────────────────────────────────
# This server runs as a web service any client can connect to

http_server_code = '''
#!/usr/bin/env python3
# tox_http_server.py -- MCP over HTTP/SSE
# Run: uvicorn tox_http_server:app --host 0.0.0.0 --port 8000
# Connect: http://localhost:8000/sse

from mcp.server import Server
from mcp.server.sse import SseServerTransport
from mcp import types
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
import json

# Create FastAPI app and MCP server
app    = FastAPI(title="Toxicology MCP Server", version="1.0.0")
server = Server("tox-http-server")
sse    = SseServerTransport("/messages")

# ── Health check endpoint ─────────────────────────────────────────────────
@app.get("/health")
async def health():
    return {"status": "ok", "server": "toxicology-mcp", "version": "1.0.0"}

# ── SSE endpoint (MCP connection) ────────────────────────────────────────
@app.get("/sse")
async def sse_endpoint(request: Request):
    async with sse.connect_sse(request.scope, request.receive, request._send) as streams:
        await server.run(streams[0], streams[1], server.create_initialization_options())

@app.post("/messages")
async def handle_messages(request: Request):
    await sse.handle_post_message(request.scope, request.receive, request._send)

# ── Tools ────────────────────────────────────────────────────────────────
@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name="pubchem_lookup",
            description="Look up a compound in PubChem by name to get CID, SMILES, and properties",
            inputSchema={
                "type": "object",
                "properties": {
                    "compound_name": {"type": "string", "description": "Chemical name"}
                },
                "required": ["compound_name"]
            }
        ),
        types.Tool(
            name="compute_herg_risk",
            description="Predict hERG cardiac toxicity risk from SMILES (CiPA Tier 1)",
            inputSchema={
                "type": "object",
                "properties": {
                    "smiles": {"type": "string"},
                    "free_cmax_um": {"type": "number", "description": "Free plasma Cmax in uM"}
                },
                "required": ["smiles"]
            }
        )
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "pubchem_lookup":
        # In production: call httpx.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/JSON")
        result = pubchem_lookup_simulated(arguments["compound_name"])
    elif name == "compute_herg_risk":
        result = herg_risk_model(arguments["smiles"], arguments.get("free_cmax_um", 0.1))
    else:
        result = {"error": f"Unknown tool: {name}"}
    return [types.TextContent(type="text", text=json.dumps(result, indent=2))]

# ── Simulated PubChem lookup (replace with real API call) ─────────────────
def pubchem_lookup_simulated(name):
    DB = {
        "aspirin": {"cid": 2244, "smiles": "CC(=O)Oc1ccccc1C(=O)O", "mw": 180.16},
        "caffeine": {"cid": 2519, "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "mw": 194.19},
    }
    return DB.get(name.lower(), {"error": f"Not found: {name}"})

def herg_risk_model(smiles, cmax):
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return {"error": "Invalid SMILES"}
    logp  = Descriptors.MolLogP(mol)
    basic = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    naro  = rdMolDescriptors.CalcNumAromaticRings(mol)
    score = 0.3*max(0,logp-2) + 0.15*basic + 0.1*naro
    risk  = "HIGH" if score>2 else "MEDIUM" if score>1 else "LOW"
    ic50  = 10**(2 - score)  # rough estimate
    margin = ic50 / cmax if cmax > 0 else float("inf")
    return {
        "herg_risk": risk, "estimated_ic50_um": round(ic50, 2),
        "free_cmax_um": cmax, "safety_margin": round(margin, 1),
        "ich_s7b_flag": margin < 10,
        "recommendation": "Proceed to CiPA Tier 2" if risk in ["HIGH","MEDIUM"] else "No further cardiac testing"
    }
'''

with open('tox_http_server.py', 'w') as f:
    f.write(http_server_code)
print('Saved: tox_http_server.py')
print()
print('To run the HTTP server:')
print('  pip install uvicorn fastapi')
print('  uvicorn tox_http_server:app --host 0.0.0.0 --port 8000')
print()
print('To connect Claude to HTTP server (claude_desktop_config.json):')
print('  {')
print('    "mcpServers": {')
print('      "tox-http": {')
print('        "url": "http://localhost:8000/sse"')
print('      }')
print('    }')
print('  }')

---
## Section 6 -- Complete ADMET Toxicology MCP Server

This is the production-ready server combining all tools into one package.
It covers the full NAM (New Approach Methods) screening pipeline.

### What this server replaces

| Tool | Animal experiment replaced |
|------|---------------------------|
| `compute_admet` | Oral bioavailability rat study |
| `screen_alerts` | Ames in vitro + in vivo |
| `predict_herg` | hERG patch-clamp + rabbit QT |
| `predict_dili` | 28-day rat hepatotoxicity |
| `compute_ivive` | Dose-range finding animal study |
| `assess_risk` | Full regulatory battery |

In [ ]:
# ── Complete ADMET server -- all tools in one file ──────────────────────────

full_server_code = '''
#!/usr/bin/env python3
# complete_admet_server.py
# Full ADMET toxicology MCP server
# Run: python complete_admet_server.py

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import json
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog

server = Server("complete-admet-tox")

@server.list_tools()
async def list_tools():
    return [

        types.Tool(
            name="compute_admet",
            description="Compute full ADMET physicochemical profile (MW, LogP, TPSA, HBD, HBA, QED, Ro5, BBB estimate, solubility class)",
            inputSchema={"type":"object","properties":{"smiles":{"type":"string"}},"required":["smiles"]}
        ),

        types.Tool(
            name="screen_genotoxicity",
            description="ICH M7(R2) two-method genotoxicity assessment: structural alerts (Method 1) + QSAR score (Method 2). Returns ICH class and recommendation.",
            inputSchema={"type":"object","properties":{"smiles":{"type":"string"}},"required":["smiles"]}
        ),

        types.Tool(
            name="predict_herg",
            description="CiPA Tier 1 hERG cardiac risk: estimate IC50, safety margin vs Cmax, TdP risk tier",
            inputSchema={
                "type":"object",
                "properties":{
                    "smiles":{"type":"string"},
                    "free_cmax_um":{"type":"number","description":"Free plasma Cmax in uM","default":0.1}
                },
                "required":["smiles"]
            }
        ),

        types.Tool(
            name="predict_dili",
            description="Predict Drug-Induced Liver Injury (DILI) risk based on structural features and physicochemical properties",
            inputSchema={"type":"object","properties":{"smiles":{"type":"string"}},"required":["smiles"]}
        ),

        types.Tool(
            name="compute_ivive",
            description="EPA HTTK IVIVE: convert in vitro EC50 to in vivo AED (Administered Equivalent Dose) and compare to ICH M7 TTC",
            inputSchema={
                "type":"object",
                "properties":{
                    "smiles":{"type":"string"},
                    "ec50_um":{"type":"number","description":"In vitro EC50 in uM"},
                    "dose_mg_day":{"type":"number","description":"Intended daily dose in mg/day","default":100}
                },
                "required":["smiles","ec50_um"]
            }
        ),

        types.Tool(
            name="full_risk_assessment",
            description="Run complete animal-free toxicology risk assessment covering all ADMET endpoints. Returns IATA WoE risk tier and 3Rs justification.",
            inputSchema={
                "type":"object",
                "properties":{
                    "smiles":{"type":"string"},
                    "compound_name":{"type":"string","description":"Compound name for report"},
                    "free_cmax_um":{"type":"number","default":0.1},
                    "ec50_um":{"type":"number","default":1.0}
                },
                "required":["smiles"]
            }
        ),

    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    try:
        smiles = arguments.get("smiles","")
        if name == "compute_admet":
            result = _admet(smiles)
        elif name == "screen_genotoxicity":
            result = _genotox(smiles)
        elif name == "predict_herg":
            result = _herg(smiles, arguments.get("free_cmax_um", 0.1))
        elif name == "predict_dili":
            result = _dili(smiles)
        elif name == "compute_ivive":
            result = _ivive(smiles, arguments["ec50_um"], arguments.get("dose_mg_day",100))
        elif name == "full_risk_assessment":
            result = _full_assessment(
                smiles,
                arguments.get("compound_name","Unknown"),
                arguments.get("free_cmax_um",0.1),
                arguments.get("ec50_um",1.0)
            )
        else:
            result = {"error": f"Unknown tool: {name}"}
        return [types.TextContent(type="text", text=json.dumps(result, indent=2))]
    except Exception as e:
        return [types.TextContent(type="text", text=json.dumps({"error": str(e)}))]

# ── Science functions ────────────────────────────────────────────────────

def _mol(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: raise ValueError(f"Invalid SMILES: {smi}")
    return mol

def _admet(smi):
    mol = _mol(smi)
    mw=Descriptors.MolWt(mol); logp=Descriptors.MolLogP(mol)
    tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)
    hba=rdMolDescriptors.CalcNumHBA(mol); rb=rdMolDescriptors.CalcNumRotatableBonds(mol)
    ro5=sum([mw>500,logp>5,hbd>5,hba>10])
    return {
        "MW":round(mw,2),"LogP":round(logp,3),"TPSA":round(tpsa,1),
        "HBD":hbd,"HBA":hba,"RotBonds":rb,"QED":round(QED.qed(mol),3),
        "Ro5_violations":ro5,"oral_BA":"Likely" if ro5<=1 else "Poor",
        "BBB_estimate":"Penetrant" if tpsa<90 and 1<logp<3 and mw<400 else "Non-CNS",
        "solubility":"Good" if logp<3 else "Moderate" if logp<5 else "Poor",
    }

def _genotox(smi):
    mol = _mol(smi)
    ALERTS={"Nitrosamine":"[N;!$(N=O)]-N=O","Ar_nitro":"c[N+](=O)[O-]",
            "Ar_amine":"[NH2]c","Michael":"[$(C=CC=O)]","Epoxide":"[C;R0]1OC1"}
    hits=[k for k,v in ALERTS.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
    c1=[h for h in hits if "Nitrosamine" in h]
    ich=("Class 1" if c1 else "Class 2" if hits else "Class 5")
    return {"alerts":hits,"ich_m7_class":ich,
            "recommendation":"Do not progress" if c1 else ("Genotoxicity testing required" if hits else "No further testing needed"),
            "replaces":"Ames test + micronucleus (ICH M7 R2)"}

def _herg(smi, cmax=0.1):
    mol = _mol(smi)
    logp=Descriptors.MolLogP(mol)
    basic=sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    naro=rdMolDescriptors.CalcNumAromaticRings(mol)
    score=0.3*max(0,logp-2)+0.15*basic+0.1*naro
    risk="HIGH" if score>2 else "MEDIUM" if score>1 else "LOW"
    ic50=round(10**(2-score),2)
    margin=round(ic50/cmax,1) if cmax>0 else 999
    return {"herg_risk":risk,"estimated_ic50_um":ic50,
            "safety_margin_x":margin,"ich_flag":margin<10,
            "cipa_tier1_score":round(score,3),
            "next_step":"CiPA Tier 2 hiPS-CM MEA" if risk in ["HIGH","MEDIUM"] else "No further cardiac testing",
            "replaces":"Rabbit in vivo QT study (ICH E14/S7B 2022)"}

def _dili(smi):
    mol = _mol(smi)
    logp=Descriptors.MolLogP(mol); mw=Descriptors.MolWt(mol)
    DILI_ALERTS={"Quinone":"O=C1C=CC(=O)C=C1","Michael":"[$(C=CC=O)]",
                 "Furan":"c1ccoc1","Nitro_aro":"c[N+](=O)[O-]"}
    nalerts=sum(1 for k,v in DILI_ALERTS.items()
                if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v)))
    lipotox=(logp>3 and mw>400)
    risk="HIGH" if nalerts>=2 or (nalerts>=1 and lipotox) else "MODERATE" if nalerts>=1 or lipotox else "LOW"
    return {"dili_risk":risk,"n_dili_alerts":nalerts,"lipophilic_MW_flag":lipotox,
            "replaces":"28-day rat hepatotoxicity study (FDA Modernization Act 2.0)"}

def _ivive(smi, ec50, dose_mg_day=100):
    mol = _mol(smi)
    logp=Descriptors.MolLogP(mol); mw=Descriptors.MolWt(mol)
    fup=max(0.001,min(1,10**(-0.028*logp-0.0038*mw+1.2)))
    import math, random; random.seed(hash(smi)%9999)
    clint=max(0.1,10**(0.35*logp-0.002*mw+random.gauss(0,0.3)))
    clint_L=clint*45*1500/1000; Qh=90
    CLh=(Qh*clint_L*fup)/(Qh+clint_L*fup)
    bw=70
    AED_mg_kg=(ec50*CLh/1000*mw*1440)/(bw*1000)
    ttc=AED_mg_kg*1000*bw*1000
    intake_ng=(dose_mg_day*1e6)*1e-9 if dose_mg_day else 0
    return {"fup":round(fup,4),"CLint_mL_min_mg":round(clint,2),
            "CLh_L_h":round(CLh,2),"AED_mg_kg_day":round(AED_mg_kg,6),
            "TTC_ug_day":round(ttc,3),"TTC_flag":ttc<1.5,
            "method":"EPA HTTK IVIVE","replaces":"Dose-range finding animal study"}

def _full_assessment(smi, name, cmax, ec50):
    admet = _admet(smi); geno = _genotox(smi)
    herg  = _herg(smi, cmax); dili = _dili(smi)
    ivive = _ivive(smi, ec50)
    concerns=[]
    if geno["ich_m7_class"] != "Class 5": concerns.append(f"Genotox: {geno['ich_m7_class']}")
    if herg["herg_risk"] in ["HIGH","MEDIUM"]: concerns.append(f"hERG: {herg['herg_risk']}")
    if dili["dili_risk"] != "LOW": concerns.append(f"DILI: {dili['dili_risk']}")
    if ivive["TTC_flag"]: concerns.append(f"TTC: {ivive['TTC_ug_day']} ug/day < 1.5 threshold")
    risk=("HIGH" if len(concerns)>=3 else "MODERATE" if len(concerns)>=1 else "LOW")
    return {
        "compound":name,"overall_risk":risk,"n_concerns":len(concerns),
        "concerns":concerns,"admet":admet,"genotoxicity":geno,
        "cardiac":herg,"hepatic":dili,"ivive":ivive,
        "3rs_justified":risk=="LOW",
        "recommendation":("No animal studies required -- NAM data sufficient"
                           if risk=="LOW" else "Targeted follow-up required"),
        "regulatory_basis":["ICH M7(R2)","ICH E14/S7B 2022","EPA HTTK",
                              "FDA Modernization Act 2.0","OECD GD 255"]
    }

async def main():
    async with stdio_server() as (r, w):
        await server.run(r, w, server.create_initialization_options())

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
'''

with open('complete_admet_server.py', 'w') as f:
    f.write(full_server_code)
print('Saved: complete_admet_server.py')
print()
print('6 tools exposed:')
tools = ['compute_admet','screen_genotoxicity','predict_herg','predict_dili','compute_ivive','full_risk_assessment']
for t in tools:
    print(f'  {t}')

---
## Section 7 -- Using MCP: Calling Your Server Programmatically

You can call an MCP server directly from Python using the MCP client library.
This lets you test your server, build pipelines, or call it from notebooks.

```
Notebook / Script
       |
       | MCP Client
       |
    MCP Server (complete_admet_server.py)
       |
    RDKit / Models / Databases
```

In [ ]:
# ── MCP Client: calling tools from Python ────────────────────────────────────
# You can call MCP tools directly without running a separate server
# by importing and calling the science functions directly.
# This is useful for testing and notebook use.

import warnings; warnings.filterwarnings('ignore')

# ── Option A: Direct function calls (notebook testing) ──────────────────────
# Call the same functions used by the server

TEST_COMPOUNDS = [
    ('Aspirin',   'CC(=O)Oc1ccccc1C(=O)O'),
    ('Cisapride', 'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1'),
    ('NDMA',      'CN(C)N=O'),
    ('Caffeine',  'Cn1cnc2c1c(=O)n(C)c(=O)n2C'),
]

print('Running full risk assessment via direct function calls:')
print('(Same logic as MCP server tools)')
print('='*65)

for name, smiles in TEST_COMPOUNDS:
    result = _full_assessment(smiles, name, cmax=0.1, ec50=1.0)
    risk = result['overall_risk']
    icons = {'LOW':'[OK]', 'MODERATE':'[!]', 'HIGH':'[!!]', 'CRITICAL':'[!!!]'}
    print(f"\n{icons.get(risk,'[?]')} {name} -- Risk: {risk}")
    print(f"   ADMET: MW={result['admet']['MW']}  LogP={result['admet']['LogP']}  QED={result['admet']['QED']}")
    print(f"   Genotox: {result['genotoxicity']['ich_m7_class']}")
    print(f"   hERG: {result['cardiac']['herg_risk']}  DILI: {result['hepatic']['dili_risk']}")
    print(f"   TTC: {result['ivive']['TTC_ug_day']} ug/day")
    if result['concerns']:
        print(f"   Concerns: {result['concerns']}")
    print(f"   3Rs: {result['recommendation'][:55]}")

In [ ]:
# ── Option B: Async MCP Client (connects to running server) ─────────────────
# This shows how to connect to your server as a real MCP client

mcp_client_code = '''
#!/usr/bin/env python3
# mcp_client_example.py
# Shows how to connect to and call an MCP server

import asyncio, json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    # Connect to the server
    server_params = StdioServerParameters(
        command="python",
        args=["complete_admet_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:

            # Step 1: Initialize
            await session.initialize()
            print("Connected to MCP server")

            # Step 2: List available tools
            tools = await session.list_tools()
            print(f"Available tools ({len(tools.tools)}):")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description[:50]}")

            # Step 3: Call a tool
            result = await session.call_tool(
                "full_risk_assessment",
                arguments={
                    "smiles": "CC(=O)Oc1ccccc1C(=O)O",
                    "compound_name": "Aspirin",
                    "free_cmax_um": 0.5,
                    "ec50_um": 1.0
                }
            )

            # Step 4: Use the result
            data = json.loads(result.content[0].text)
            print(f"\nAspirin risk assessment:")
            print(f"  Overall risk:  {data['overall_risk']}")
            print(f"  3Rs justified: {data['3rs_justified']}")
            print(f"  Recommendation: {data['recommendation']}")

if __name__ == "__main__":
    asyncio.run(main())
'''

with open('mcp_client_example.py', 'w') as f:
    f.write(mcp_client_code)
print('Saved: mcp_client_example.py')
print()
print('To run:')
print('  python mcp_client_example.py')
print()
print('What this client does step by step:')
print('  1. Opens subprocess running complete_admet_server.py')
print('  2. Sends MCP initialize handshake')
print('  3. Lists available tools')
print('  4. Calls full_risk_assessment with Aspirin SMILES')
print('  5. Parses and prints the result')

---
## Section 8 -- Connecting Claude Desktop to Your MCP Server

Claude Desktop supports MCP natively. Once configured, Claude can call your
toxicology tools directly in conversation -- no code needed from the user.

### Step-by-step: Connecting Claude Desktop

**Step 1:** Find your config file
```
macOS:   ~/Library/Application Support/Claude/claude_desktop_config.json
Windows: %APPDATA%/Claude/claude_desktop_config.json
Linux:   ~/.config/Claude/claude_desktop_config.json
```

**Step 2:** Add your server
```json
{
  "mcpServers": {
    "toxicology": {
      "command": "python",
      "args": ["/absolute/path/to/complete_admet_server.py"],
      "env": {}
    }
  }
}
```

**Step 3:** Restart Claude Desktop

**Step 4:** Look for the hammer icon -- your tools appear automatically

**Step 5:** Ask Claude:
> *'Can you run a full toxicology risk assessment on aspirin (SMILES: CC(=O)Oc1ccccc1C(=O)O)?'

Claude will automatically call your `full_risk_assessment` tool and interpret the results.

### What Claude can now do with your server

```
User: Is diclofenac hepatotoxic?
Claude: [calls predict_dili with diclofenac SMILES]
        [calls compute_admet with diclofenac SMILES]
        Based on my analysis: DILI risk is HIGH. LogP=3.96 and
        MW=296 suggest lipophilic hepatic accumulation. This
        compound is classified Most-concern in FDA DILIrank...

User: Screen these 5 compounds for ICH M7 genotoxicity
Claude: [calls screen_genotoxicity for each compound]
        [assembles comparative table]
        Here are the ICH M7 assessments...
```

In [ ]:
# ── Auto-generate the Claude Desktop config ──────────────────────────────────
import os, json

# Get absolute path to the server file
server_path = os.path.abspath('complete_admet_server.py')

config = {
    'mcpServers': {
        'toxicology-admet': {
            'command': 'python',
            'args': [server_path],
            'env': {}
        },
        'tox-database': {
            'command': 'python',
            'args': [os.path.abspath('tox_database_server.py')],
            'env': {}
        }
    }
}

# Save config
with open('claude_desktop_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Generated claude_desktop_config.json')
print()
print('Content:')
print(json.dumps(config, indent=2))
print()
print('Copy this to your Claude Desktop config location:')
print('  macOS:   ~/Library/Application Support/Claude/claude_desktop_config.json')
print('  Windows: %APPDATA%/Claude/claude_desktop_config.json')
print()
print('After restarting Claude Desktop, you will see a hammer icon')
print('indicating MCP tools are available.')

---
## Section 9 -- Using MCP Tools with the Anthropic API

You can also call MCP tools programmatically using the Anthropic API
(anthropic SDK). This lets you build automated toxicology pipelines.

```
Your Python script
       |
       | anthropic.messages.create(tools=[...])
       |
    Claude API
       |
       | tool_use block returned
       |
Your script calls your MCP tools locally
       |
       | tool_result sent back
       |
    Claude API gives final answer
```

This is the pattern for building **automated screening pipelines** where
Claude orchestrates the toxicology analysis without human intervention.

In [ ]:
# ── Anthropic API + MCP tools pattern ───────────────────────────────────────
# This shows how to use Claude as an AI orchestrator that calls your tools

anthropic_pipeline_code = '''
#!/usr/bin/env python3
# anthropic_tox_pipeline.py
# Claude orchestrates toxicology analysis using your MCP tools
# Requires: pip install anthropic
# Set:      export ANTHROPIC_API_KEY=your_key_here

import anthropic, json
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# Define your toxicology tools in Anthropic format
# (same logic as MCP server, different format for direct API use)
TOOLS = [
    {
        "name": "compute_admet",
        "description": "Compute ADMET properties for a molecule: MW, LogP, TPSA, HBD, HBA, QED, Ro5",
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {"type": "string", "description": "SMILES of the molecule"}
            },
            "required": ["smiles"]
        }
    },
    {
        "name": "screen_genotoxicity",
        "description": "Screen for ICH M7 structural alerts and QSAR genotoxicity prediction",
        "input_schema": {
            "type": "object",
            "properties": {"smiles": {"type": "string"}},
            "required": ["smiles"]
        }
    },
    {
        "name": "predict_herg",
        "description": "Predict hERG cardiac risk (CiPA Tier 1)",
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {"type": "string"},
                "free_cmax_um": {"type": "number"}
            },
            "required": ["smiles"]
        }
    }
]

# Local tool executor (calls your actual science functions)
def execute_tool(name, inputs):
    mol = Chem.MolFromSmiles(inputs.get("smiles", ""))
    if not mol: return {"error": "Invalid SMILES"}
    if name == "compute_admet":
        return {"MW": round(Descriptors.MolWt(mol),2),
                "LogP": round(Descriptors.MolLogP(mol),3),
                "QED": round(QED.qed(mol),3)}
    if name == "screen_genotoxicity":
        ALERTS={"Nitrosamine":"[N;!$(N=O)]-N=O","Ar_nitro":"c[N+](=O)[O-]"}
        hits=[k for k,v in ALERTS.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
        return {"alerts": hits, "ich_m7_class": "Class 1" if "Nitrosamine" in hits else ("Class 2" if hits else "Class 5")}
    if name == "predict_herg":
        logp=Descriptors.MolLogP(mol)
        risk="HIGH" if logp>4 else "MEDIUM" if logp>2 else "LOW"
        return {"herg_risk": risk, "estimated_ic50_um": round(10**(2-logp*0.3),2)}
    return {"error": f"Unknown tool: {name}"}

def run_tox_analysis(compound_name, smiles):
    print(f"Analyzing: {compound_name}")
    messages = [{"role": "user", "content": f"Please run a complete toxicology assessment for {compound_name} (SMILES: {smiles}). Use all available tools."}]

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2000,
            tools=TOOLS,
            messages=messages
        )

        if response.stop_reason == "tool_use":
            # Claude wants to call tools
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Calling: {block.name}({block.input})")
                    result = execute_tool(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,
                                         "content":json.dumps(result)})
            # Add tool results and continue
            messages.append({"role":"assistant","content":response.content})
            messages.append({"role":"user","content":tool_results})
        else:
            # Final answer
            final = next(b.text for b in response.content if hasattr(b,"text"))
            print(f"  Result: {final[:200]}...")
            return final

# Run analysis
# run_tox_analysis("Aspirin", "CC(=O)Oc1ccccc1C(=O)O")
# run_tox_analysis("NDMA",    "CN(C)N=O")
print("Uncomment run_tox_analysis() calls above to run (requires ANTHROPIC_API_KEY)")
'''

with open('anthropic_tox_pipeline.py', 'w') as f:
    f.write(anthropic_pipeline_code)
print('Saved: anthropic_tox_pipeline.py')
print()
print('This shows the agentic pattern:')
print('  1. Send question to Claude')
print('  2. Claude decides which tools to call')
print('  3. Your code executes the tools')
print('  4. Results go back to Claude')
print('  5. Claude synthesises a regulatory assessment')

---
## Section 10 -- Best Practices, Patterns & Complete Reference

### Common MCP patterns in computational toxicology

**Pattern 1: Tiered screening (fast to slow)**
```
Tool 1: structural_alert_screen (milliseconds, RDKit)
           |
           v if alerts found
Tool 2: qsar_prediction (seconds, sklearn model)
           |
           v if positive
Tool 3: organoid_assay_query (minutes, database lookup)
```

**Pattern 2: Parallel tool calls**
```
Claude can call multiple tools simultaneously:
  compute_admet(smiles)        <-- parallel
  screen_genotoxicity(smiles)  <-- parallel
  predict_herg(smiles)         <-- parallel
  All finish, Claude assembles IATA report
```

**Pattern 3: Tool chaining**
```
pubchem_lookup(name) --> get SMILES
     |
     v
compute_admet(smiles) --> get properties
     |
     v
generate_iata_report(all_data) --> regulatory output
```

### Error handling rules

| Error type | What to do | Example |
|-----------|-----------|--------|
| Invalid SMILES | Return error JSON, never crash | `{'error': 'Invalid SMILES: ...'}`|
| Missing argument | Check before calling | `if 'smiles' not in arguments:` |
| Model exception | Catch and return gracefully | `try/except` every tool |
| Unknown tool name | Return clear error | `raise ValueError(f'Unknown: {name}')` |

In [ ]:
# ── Best practices demonstrated ───────────────────────────────────────────

import json

print('MCP Toxicology Server -- Best Practices Demo')
print('='*55)

# 1. Always validate SMILES before any computation
def safe_smiles(smiles: str):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, {'error': f'Invalid SMILES: {smiles!r}', 'action': 'Check SMILES notation'}
    return mol, None

print('1. SMILES validation:')
mol, err = safe_smiles('CC(=O)Oc1ccccc1C(=O)O')
print(f'   Valid:   mol={mol is not None}')
mol, err = safe_smiles('NOT_A_SMILES')
print(f'   Invalid: error={err}')

# 2. Always return structured JSON, never raw text
print()
print('2. Structured output format:')
good_output = {
    'compound': 'Aspirin',
    'MW': 180.16,
    'risk': 'LOW',
    'regulatory_basis': 'ICH M7(R2)',
    'recommendation': 'No further testing',
}
bad_output = 'Aspirin MW is 180 and risk is low based on ICH M7.'
print(f'   Good (structured): {json.dumps(good_output)[:80]}')
print(f'   Bad  (raw text):   {bad_output}')

# 3. Include metadata in every response
print()
print('3. Always include metadata:')
def add_metadata(result: dict, tool_name: str, replaces: str) -> dict:
    return {
        **result,
        '_metadata': {
            'tool': tool_name,
            'replaces_animal_test': replaces,
            'regulatory_frameworks': ['OECD GD 69', 'FDA Modernization Act 2.0'],
        }
    }
result = add_metadata({'MW': 180.16, 'risk': 'LOW'}, 'compute_admet', 'Oral bioavailability rat study')
print(f'   With metadata: {json.dumps(result)[:100]}...')

# 4. Tool description quality
print()
print('4. Tool description quality:')
bad_desc  = 'Computes stuff for molecules'
good_desc = ('Compute ADMET physicochemical properties from SMILES: '
             'MW, LogP, TPSA, HBD, HBA, QED, Ro5 violations, oral bioavailability estimate. '
             'Replaces oral bioavailability rat study. Based on Lipinski Ro5 and Veber rules.')
print(f'   Bad:  {bad_desc!r}')
print(f'   Good: {good_desc[:100]}...')

In [ ]:
# ── Complete file listing and setup summary ──────────────────────────────────
import os

files = [
    ('my_first_server.py',          'Hello World MCP server (Section 2)'),
    ('rdkit_mcp_server.py',         'RDKit cheminformatics tools (Section 3)'),
    ('tox_database_server.py',      'Database resources + search (Section 4)'),
    ('tox_http_server.py',          'HTTP/SSE transport for teams (Section 5)'),
    ('complete_admet_server.py',    'Production ADMET server -- use this (Section 6)'),
    ('mcp_client_example.py',       'Python MCP client example (Section 7)'),
    ('anthropic_tox_pipeline.py',   'Anthropic API agentic pipeline (Section 9)'),
    ('claude_desktop_config.json',  'Claude Desktop configuration (Section 8)'),
]

print('Files created by this tutorial:')
print('='*60)
for filename, description in files:
    exists = os.path.exists(filename)
    status = 'OK' if exists else 'missing'
    print(f'  [{status:7s}] {filename:35s} {description}')

print()
print('Quick start (3 steps to get running):')
print()
print('Step 1: Install dependencies')
print('  pip install mcp rdkit fastapi uvicorn httpx')
print()
print('Step 2: Test the server')
print('  python mcp_client_example.py')
print()
print('Step 3: Connect to Claude Desktop')
print('  Copy claude_desktop_config.json to your Claude config directory')
print('  Restart Claude Desktop')
print('  Ask: Run full toxicology risk assessment on aspirin')
print()
print('What Claude can now do for you:')
tasks = [
    'Screen 100 compounds for ICH M7 genotoxicity',
    'Generate regulatory IATA reports automatically',
    'Compare ADMET profiles across a drug series',
    'Flag hERG cardiac risks before synthesis',
    'Calculate IVIVE doses for 3Rs compliance',
    'Search toxicology database by hazard class',
]
for t in tasks:
    print(f'  * {t}')

In [ ]:
# ── Cheatsheet ────────────────────────────────────────────────────────────────
lines = [
    'MCP TOXICOLOGY QUICK REFERENCE',
    '',
    'CORE CONCEPTS',
    '  Server     -- Python script exposing tools/resources',
    '  Tool       -- function Claude can call (active)',
    '  Resource   -- data Claude can read (passive)',
    '  Prompt     -- template prompt Claude can use',
    '  Transport  -- STDIO (local) or HTTP/SSE (network)',
    '',
    'MINIMAL SERVER STRUCTURE',
    '  server = Server("my-server")',
    '  @server.list_tools() async def list_tools(): return [...]',
    '  @server.call_tool()  async def call_tool(name, args): ...',
    '  stdio_server() + server.run()  -- starts the server',
    '',
    'TOOL DEFINITION (must include)',
    '  name:         snake_case, unique',
    '  description:  what it does, what it replaces, guidelines',
    '  inputSchema:  JSON Schema with types + required fields',
    '',
    'TOXICOLOGY TOOLS PATTERN',
    '  compute_admet(smiles)           --> ADMET properties',
    '  screen_genotoxicity(smiles)     --> ICH M7 two-method',
    '  predict_herg(smiles, cmax)      --> CiPA Tier 1',
    '  predict_dili(smiles)            --> DILI risk',
    '  compute_ivive(smiles, ec50)     --> EPA HTTK AED',
    '  full_risk_assessment(smiles)    --> IATA WoE report',
    '',
    'CLAUDE DESKTOP CONFIG',
    '  Location: ~/Library/Application Support/Claude/claude_desktop_config.json',
    '  Format: {"mcpServers": {"name": {"command": "python", "args": ["server.py"]}}}',
    '  After adding: restart Claude Desktop',
    '',
    'ANTHROPIC API PATTERN',
    '  client.messages.create(tools=[TOOLS], messages=[...])',
    '  if stop_reason == "tool_use": execute tools locally',
    '  send tool_result back, get final answer',
    '',
    'ERROR HANDLING',
    '  Always validate SMILES before RDKit calls',
    '  Wrap call_tool body in try/except',
    '  Return {"error": msg} not raise Exception',
    '  Never return None -- always return structured JSON',
    '',
    'REGULATORY ALIGNMENT',
    '  ICH M7(R2) 2023   -- genotoxicity two-method',
    '  ICH E14/S7B 2022  -- hERG/CiPA cardiac',
    '  EPA HTTK          -- IVIVE dosimetry',
    '  OECD GD 69        -- QSAR model validation',
    '  FDA Mod Act 2.0   -- NAM data accepted',
]
print('\n'.join(lines))